# VINE: four inpainting methods

Repository: [https://github.com/Shilin-LU/VINE](https://github.com/Shilin-LU/VINE)

Paper: [https://arxiv.org/abs/2410.18775](https://arxiv.org/abs/2410.18775)

Inspected commit: `d72f11a93f8de9bc45ef61202b9c8dda2a70cc77`

Watermark stage: `post_hoc`
Domain: image (framewise video adapter)

This method can be applied to existing media, so the notebook uses the fixed MOSE sample.

ProPainter, DiffuEraser, EffectErase, and E2FGVI-HQ receive the same authentic watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for that pipeline, then both reported outputs are trimmed to the authentic length. EffectErase applies its released Wan 4n+1 temporal adapter; E2FGVI-HQ uses the authors' official arbitrary-resolution test entry point.

VINE's official list encoder center-crops non-square images. The adapter applies that exact crop to every MOSE frame and mask, embeds one fixed 100-bit text payload, and records the geometry. The released decoder has no pixel localizer, so only bit recovery is reported.


## Authenticity contract

- Watermark preparation: Extract the video frames, run the official VINE-R list encoder with one fixed message, and reassemble the encoded frames at the source frame rate.
- Checkpoint status: READY: VINE-R and VINE-B encoder/decoder checkpoints are published on Hugging Face.
- Protocol note: VINE center-crops non-square inputs in its demo.  The experiment must record the spatial adapter and align the object-removal mask to the watermarked frame geometry.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))
from watermark_notebook_runtime import configured_path, require_commit, require_path

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

require = require_path

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in (
        "input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4",
        "effecterase.mp4", "e2fgvi-browser.mp4",
    ):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    result_json = {
        *run_root.rglob("summary.json"),
        *(attack / name for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json")),
    }
    for path in sorted(path for path in result_json if path.is_file()):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


## Canonical one-clip comparison: all four inpainters

This is the canonical comparison for the minimum one-clip requirement. All four removers use the same frozen watermarked input and aligned binary mask. Every remover is presented in the same order, with the same output-level fields and the same video/JSON evidence. Method-specific resizing, padding, and mask dilation remain recorded in JSON rather than being silently normalized.

| Inpainting method | Status | Runtime (s) | Frames | Dimensions | FPS | Bytes | Evidence |
|---|---:|---:|---:|---:|---:|---:|---|
| ProPainter | complete | not recorded | 67 | 480 × 480 | 25.000 | 437,480 | [video](../../runs/remaining-watermarks/vine/02b72b8d/attack/propainter.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/propainter.json) |
| DiffuEraser | complete | not recorded | 67 | 480 × 480 | 25.000 | 532,990 | [video](../../runs/remaining-watermarks/vine/02b72b8d/attack/diffueraser.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/diffueraser.json) |
| EffectErase | complete | 115.154 | 67 | 480 × 480 | 25.000 | 252,569 | [video](../../runs/remaining-watermarks/vine/02b72b8d/attack/effecterase.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/effecterase.json) |
| E2FGVI-HQ | complete | 14.032 | 67 | 480 × 480 | 25.000 | 644,161 | [playable video](../../runs/remaining-watermarks/vine/02b72b8d/attack/e2fgvi-browser.mp4) · [evaluated MP4](../../runs/remaining-watermarks/vine/02b72b8d/attack/e2fgvi.mp4) · [JSON](../../runs/remaining-watermarks/vine/02b72b8d/attack/e2fgvi.json) |

Watermark method: **Vine**. Runtime is shown only when the original run record captured it; `not recorded` is not replaced with an estimate. Watermark-specific localization or recovery metrics are downstream evaluations and are kept separate from this equal inpainting comparison. Where an evaluated codec is not browser-safe, the playable video is an H.264 viewing copy; the evaluated MP4 and exact JSON record remain unchanged.


In [ ]:
from pathlib import Path
import json
from IPython.display import JSON, Markdown, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/finalize_four_inpainter_notebooks.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
ATTACK = ROOT / 'runs/remaining-watermarks/vine/02b72b8d/attack'
for name in ("input-watermarked.mp4", "watermarked.mp4", "gt-mask.mp4"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(Video(str(path), embed=True, html_attributes="controls muted loop"))

REMOVERS = (
    ("ProPainter", "propainter"),
    ("DiffuEraser", "diffueraser"),
    ("EffectErase", "effecterase"),
    ("E2FGVI-HQ", "e2fgvi"),
)
for label, stem in REMOVERS:
    display(Markdown(f"### {label}"))
    evaluated_video = ATTACK / f"{stem}.mp4"
    browser_video = ATTACK / f"{stem}-browser.mp4"
    video = browser_video if browser_video.is_file() else evaluated_video
    metadata = ATTACK / f"{stem}.json"
    if not evaluated_video.is_file() or not video.is_file() or not metadata.is_file():
        raise FileNotFoundError(f"Missing {label} evidence in {ATTACK}")
    display(Video(str(video), embed=True, html_attributes="controls muted loop"))
    display(JSON(json.loads(metadata.read_text())))

for name in ("sample.json", "removal-pair.json"):
    metadata = ATTACK / name
    if metadata.is_file():
        print(name)
        display(JSON(json.loads(metadata.read_text())))


## Watermark evaluation results: all four inpainters

The downstream watermark evaluation is complete for this method. These are the exact recorded outputs for the same four canonical attack cases; they are not estimates.

| Inpainting method | Majority payload accuracy | Mean frame accuracy |
| --- | ---: | ---: |
| ProPainter | 100.00% | 99.88% |
| DiffuEraser | 100.00% | 99.99% |
| EffectErase | 82.00% | 66.55% |
| E2FGVI-HQ | 100.00% | 99.99% |

**How to read this table:** Lower accuracy means less of the watermark remained recoverable after inpainting. Where IoU is reported, higher IoU means better overlap with the effective inpainted region.

**Method note:** The released decoder recovers a global payload and does not output a spatial tamper mask.

Exact unrounded values and execution notes are in the [consolidated 44-case record](../../evidence/watermark-evaluation-44.json).


## Full-50 evaluation: key metrics

This section is generated from the machine-readable full-50 summary. It reports the required 50 samples and all four removers; incomplete GPU runs are shown as `pending`, never as zero.

Summary: `evidence/full50-watermark-evaluation-summary.json`  
SHA-256: `7c6cbbd52cd886b5237c2b060eedf4b641c277b510050f43b8541f63a8d8b347`

| Remover | Complete | Key metric aggregates (mean ± population SD; range) |
|---|---:|---|
| ProPainter | 50/50 | Video-majority bit accuracy: 0.9894 ± 0.0552; [0.6100, 1.0000]<br>Mean frame bit accuracy: 0.9670 ± 0.0649; [0.6139, 0.9999] |
| DiffuEraser | 50/50 | Video-majority bit accuracy: 0.9922 ± 0.0477; [0.6600, 1.0000]<br>Mean frame bit accuracy: 0.9809 ± 0.0562; [0.6344, 1.0000] |
| EffectErase | 50/50 | Video-majority bit accuracy: 0.8776 ± 0.1221; [0.5700, 1.0000]<br>Mean frame bit accuracy: 0.7141 ± 0.1165; [0.5228, 0.9519] |
| E2FGVI-HQ | 50/50 | Video-majority bit accuracy: 0.9944 ± 0.0337; [0.7600, 1.0000]<br>Mean frame bit accuracy: 0.9793 ± 0.0505; [0.6948, 1.0000] |

The next cell displays the complete per-sample table directly from the same summary and checks its hash before showing values.


In [ ]:
from pathlib import Path
import hashlib
import html
import json
from IPython.display import HTML, display

FULL50_METHOD = 'vine'
FULL50_SUMMARY_SHA256 = '7c6cbbd52cd886b5237c2b060eedf4b641c277b510050f43b8541f63a8d8b347'
FULL50_SUMMARY_RELATIVE = Path("evidence/full50-watermark-evaluation-summary.json")

def find_full50_summary():
    starts = [Path.cwd(), Path.cwd().resolve()]
    for start in starts:
        for parent in (start, *start.parents):
            candidate = parent / FULL50_SUMMARY_RELATIVE
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f"Could not find {FULL50_SUMMARY_RELATIVE} from {Path.cwd()}")

summary_path = find_full50_summary()
actual_hash = hashlib.sha256(summary_path.read_bytes()).hexdigest()
if actual_hash != FULL50_SUMMARY_SHA256:
    raise RuntimeError(
        f"Full-50 summary changed (expected {FULL50_SUMMARY_SHA256}, found {actual_hash}). "
        "Run scripts/update_full50_watermark_notebooks.py before trusting this table."
    )
full50 = json.loads(summary_path.read_text())["methods"][FULL50_METHOD]
metric_labels = {item["key"]: item["label"] for item in full50["key_metrics"]}
headers = ["sample", "remover", "status", *metric_labels.values(), "evidence"]
rows = []
for sample in full50["samples"]:
    for remover, result in sample["removers"].items():
        values = [
            sample["sample_id"], remover, result["status"],
            *[result["metrics"].get(key) for key in metric_labels],
            result.get("evidence"),
        ]
        cells = "".join(
            f"<td>{html.escape('' if value is None else f'{value:.6g}' if isinstance(value, float) else str(value))}</td>"
            for value in values
        )
        rows.append(f"<tr>{cells}</tr>")
display(HTML(
    "<div style='max-height:560px;overflow:auto'><table>"
    + "<thead><tr>" + "".join(f"<th>{html.escape(value)}</th>" for value in headers) + "</tr></thead>"
    + "<tbody>" + "".join(rows) + "</tbody></table></div>"
))


In [ ]:
VINE = configured_path("VINE_ROOT", description="VINE checkout")
VINE_PYTHON = configured_path("VINE_PYTHON", default=VINE / ".venv/bin/python")
DIFFUERASER = configured_path("DIFFUERASER_ROOT", description="DiffuEraser checkout")
DIFFUERASER_PYTHON = configured_path("DIFFUERASER_PYTHON", default=DIFFUERASER / ".venv/bin/python")
GPU = os.environ.get("WATERMARK_GPU", "0")
MOSE_VIDEO = ROOT / "evidence/mose-task1/review50/media/02b72b8d/original.mp4"
MOSE_MASK = ROOT / "evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4"
RUN_ROOT = ROOT / "runs/remaining-watermarks/vine/02b72b8d"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WATERMARKED_VIDEO = RUN_ROOT / "watermarked.mp4"
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"


In [ ]:
require_commit(VINE, "d72f11a93f8de9bc45ef61202b9c8dda2a70cc77")
require(VINE_PYTHON, "VINE Python")


In [ ]:
run([
    VINE_PYTHON, SCRIPTS / "vine_video.py", "embed", "--source", VINE,
    "--input-video", MOSE_VIDEO, "--input-mask", MOSE_MASK,
    "--output-video", WATERMARKED_VIDEO, "--output-mask", ALIGNED_MASK,
    "--encoder-model", "Shilin-LU/VINE-R-Enc", "--message", "Hello World!",
])


## Run all four inpainting methods on the frozen input

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. Every remover below consumes that frozen input and mask.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "VINE",
        "--sample-id", "02b72b8d", "--sample-source", "MOSE fixed sample after upstream center crop",
        "--source-kind", "mose_post_hoc",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
EFFECTERASE = configured_path("EFFECTERASE_ROOT", description="EffectErase checkout")
EFFECTERASE_PYTHON = configured_path(
    "EFFECTERASE_PYTHON", default=EFFECTERASE / ".venv/bin/python"
)
E2FGVI = configured_path("E2FGVI_ROOT", description="E2FGVI checkout")
E2FGVI_CHECKPOINT = configured_path(
    "E2FGVI_CHECKPOINT", default=E2FGVI / "release_model/E2FGVI-HQ-CVPR22.pth"
)
E2FGVI_PYTHON = configured_path("E2FGVI_PYTHON", default=DIFFUERASER_PYTHON)

for path, label in (
    (EFFECTERASE_PYTHON, "EffectErase Python"),
    (E2FGVI_CHECKPOINT, "E2FGVI-HQ checkpoint"),
    (E2FGVI_PYTHON, "E2FGVI Python"),
):
    require(path, label)

if not (ATTACK_RUN / "effecterase.json").is_file():
    effect_env = os.environ.copy(); effect_env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        EFFECTERASE_PYTHON, SCRIPTS / "run_prepared_effecterase.py",
        "--source", EFFECTERASE, "--run", ATTACK_RUN, "--device", GPU,
    ], env=effect_env)
else:
    print("EffectErase output already exists:", ATTACK_RUN / "effecterase.json")

if not (ATTACK_RUN / "e2fgvi.json").is_file():
    run([
        E2FGVI_PYTHON, SCRIPTS / "run_prepared_e2fgvi.py",
        "--source", E2FGVI, "--checkpoint", E2FGVI_CHECKPOINT,
        "--run", ATTACK_RUN, "--python", E2FGVI_PYTHON, "--device", GPU,
    ])
else:
    print("E2FGVI-HQ output already exists:", ATTACK_RUN / "e2fgvi.json")


In [ ]:
run([
    VINE_PYTHON, SCRIPTS / "vine_video.py", "detect", "--source", VINE,
    "--decoder-model", "Shilin-LU/VINE-R-Dec", "--message", "Hello World!",
    "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
    "--case", f"effecterase={ATTACK_RUN / 'effecterase.mp4'}",
    "--case", f"e2fgvi={ATTACK_RUN / 'e2fgvi.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)
